# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzramAfaq/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm choosing this lane because the starter dataset already shows a large, concrete backlog problem: most content is visible in search, a majority of that visible content is trending down, and a meaningful slice is also stale. That's not a hypothetical — it's a pile of pages someone has to triage every week, and nobody can eyeball 30,000 rows by hand. Lane 1 (signal analysis) and Lane 3 (clustering) are more exploratory and don't end in a specific action; Lane 4 (CTR scoring) is a narrower version of the same triage problem. Lane 2 is the one where I can point to a ranked list and say "an editor should look at row 1 before row 5,000," which is the kind of output this dataset seems built for. I'll confirm this by Week 4 once I've looked at the warehouse release too, but the starter numbers already make a stale/declining/visible backlog look real and worth ranking*

In [3]:

import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AzramAfaq/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), "starter CSV not found — are you at the repo root?"

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape (rows, cols):", df.shape)

Working dir: /content/flyrank-ml-internship
Loaded: data/raw/content_refresh_anonymized.csv
Shape (rows, cols): (30000, 44)


In [4]:
# Setup — load the libraries I'll use for Sections 3 and 4, and confirm the starter file is where the data skill says it is.
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape (rows, cols):", df.shape)

Loaded: data/raw/content_refresh_anonymized.csv
Shape (rows, cols): (30000, 44)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Search question: Which pages should be reviewed first for refresh, given that a page is visible in search, trending down, and hasn't been updated in a while?

Unit of analysis: a single content item (one row = one page, content_id), pseudonymized to one of 32 clients.

Decision it improves: which page an SEO editor opens next — right now that choice is likely made by memory, a spreadsheet sort, or gut feel, across a backlog too large to read end to end.

Who acts, and what they do: a content editor or SEO strategist at a client account. Given my ranked queue and reason codes, they open the highest-priority page, check it against the reason code (e.g. "visible + declining + stale"), and decide to refresh, expand, or leave it — a human makes the final call, my output only orders the queue and explains why.

Output: a ranked action queue — page, priority score, reason code, and a confidence label — not a single verdict.

Cost of a wrong call, in both directions:

False positive (I flag a page that didn't need a refresh): wasted editor hours, and if this happens often the editor stops trusting the queue and ignores it entirely.
False negative (I miss a page that's quietly declining): a real page keeps losing clicks/visibility for another cycle before anyone notices — lost traffic that compounds the longer it's missed.
Because the second kind of miss is silent and compounds, I'd rather the queue lean toward flagging borderline cases with a clear "low confidence" reason code than stay silent about them — that's a judgment call I'll revisit once I see precision@K on real data.

Why data or ML helps at all: a single hard rule ("flag if declining") already catches over half the visible pages (see Section 3) — too many to be useful on its own. The real problem is ranking within that pile using several signals at once (trend, staleness, demand, position, content type) that interact in ways that are hard to hand-write as a single if-statement. That's a scoring/ranking problem, not a "train a model and hope" problem — a transparent baseline score comes first, and a model only earns its place if it beats that baseline.

In [5]:
# Check: how big is the "obvious rule" backlog on its own, to justify why simple rules alone aren't enough?
visible = df[df["avg_position"] > 0]
down_visible = visible[visible["trend_direction"] == "down"]

print("Visible pages:", len(visible), f"({len(visible)/len(df):.1%} of all rows)")
print("Visible AND trending down:", len(down_visible), f"({len(down_visible)/len(df):.1%} of all rows)")
print("-> a single rule already flags more pages than any editor could review by hand.")


Visible pages: 28795 (96.0% of all rows)
Visible AND trending down: 16254 (54.2% of all rows)
-> a single rule already flags more pages than any editor could review by hand.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
# 2-3 real numbers from the starter dataset that justify this lane

n_total = len(df)
n_clients = df["client_id"].nunique()

visible = df[df["avg_position"] > 0]
pct_visible = len(visible) / n_total

down_visible = visible[visible["trend_direction"] == "down"]
pct_down_visible = len(down_visible) / n_total

stale = df["freshness_tier"].isin(["91-180", "181+"])
refresh_candidates = df[(df["trend_direction"] == "down") & (df["avg_position"] > 0) & stale]
pct_candidates = len(refresh_candidates) / n_total

print(f"1) Rows / clients:            {n_total:,} rows across {n_clients} pseudonymized clients")
print(f"2) Visible pages:              {len(visible):,} of {n_total:,} ({pct_visible:.1%}) have avg_position > 0")
print(f"3) Visible AND declining:      {len(down_visible):,} ({pct_down_visible:.1%} of all rows)")
print(f"4) Refresh-candidate pool:     {len(refresh_candidates):,} pages ({pct_candidates:.1%}) are")
print("                                visible + trending down + stale (>=91 days since update)")
print()
print("-> Almost every page is visible, more than half of visible pages are declining,")
print("   and roughly 1 in 5 pages overall sits in the exact overlap a refresh queue would target.")
print("   That overlap is big enough to need ranking, and specific enough to be actionable.")

1) Rows / clients:            30,000 rows across 32 pseudonymized clients
2) Visible pages:              28,795 of 30,000 (96.0%) have avg_position > 0
3) Visible AND declining:      16,254 (54.2% of all rows)
4) Refresh-candidate pool:     5,684 pages (18.9%) are
                                visible + trending down + stale (>=91 days since update)

-> Almost every page is visible, more than half of visible pages are declining,
   and roughly 1 in 5 pages overall sits in the exact overlap a refresh queue would target.
   That overlap is big enough to need ranking, and specific enough to be actionable.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim:

Observed patterns: e.g. "X% of visible pages in this sample are trending down" — a description of what's in the data, not a prediction.
Directional, decision-support signals: e.g. "pages with these characteristics have historically been more likely to be declining" — useful for ordering a queue, not a guarantee about any single page.
A ranked priority, not a verdict: my output tells an editor where to look first; it does not tell them refreshing will work.
What I will never claim:

Causal proof. This data is observational — I watched what happened, I did not run an experiment. "Declining pages that got refreshed later recovered" (if I even see that pattern) is not the same as "refreshing causes recovery." Confounders (seasonality, competitor changes, algorithm updates) are not controlled for.
"Predicting Google." I have no visibility into ranking algorithm internals. Any model only learns correlations between page-level metrics and observed outcomes in this dataset — not how search engines actually rank pages.
Guaranteed wrong-page costs. I can estimate that missed declines are more costly than false positives, but I don't have real editor-hour data or realized traffic-recovery data to put a dollar figure on it — that's a directional argument, not a calculated one.
One leakage discipline I'm committing to now: is_declining_label is derived from trend_direction, which is derived from trend_pct. Per the data skill, trend_direction and trend_pct can never be used as model features if a label is built from them — only as the label itself, or for descriptive stats like the ones in Section 3.

In [7]:
# Quick leakage self-check: confirm I know which columns are label-only, never features
label_only_columns = ["trend_direction", "trend_pct"]
candidate_feature_columns = [c for c in df.columns if c not in label_only_columns]

print("Columns reserved for the label only (never features):", label_only_columns)
print("Remaining candidate feature columns:", len(candidate_feature_columns), "of", len(df.columns))


Columns reserved for the label only (never features): ['trend_direction', 'trend_pct']
Remaining candidate feature columns: 42 of 44


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.